# Bilateral_voting vs Bilateral_voting_kxk(conn_num=3) 비교

이 노트북은 `conn_num=3`일 때 `Bilateral_voting`과 `Bilateral_voting_kxk`의 결과가 동일한지 확인합니다.  
채널 순서가 다를 수 있으므로, 직접 비교와 채널 정렬 후 비교를 모두 수행합니다.

In [37]:
# 1. 환경 설정 및 함수 로드
import autorootcwd
import importlib
import random

import numpy as np
import torch

import connect_loss as connect_loss_module

connect_loss_module = importlib.reload(connect_loss_module)
Bilateral_voting = connect_loss_module.Bilateral_voting
Bilateral_voting_kxk = connect_loss_module.Bilateral_voting_kxk

_ = autorootcwd

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    # connect_loss.py hardcodes .cuda(); make it a no-op for CPU-only validation.
    def _cpu_cuda(self, device=None, non_blocking=False, memory_format=torch.preserve_format):
        _ = (device, non_blocking, memory_format)
        return self

    torch.Tensor.cuda = _cpu_cuda


def print_tensor_info(name, tensor):
    print(
        f"{name}: shape={tuple(tensor.shape)}, device={tensor.device}, dtype={tensor.dtype}, "
        f"min={tensor.min().item():.6f}, max={tensor.max().item():.6f}, mean={tensor.mean().item():.6f}"
    )


def build_translation_matrices(height, width, num_class=1, device=None, dtype=torch.float32):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    hori_translation = torch.zeros((1, num_class, width, width), device=device, dtype=dtype)
    for i in range(width - 1):
        hori_translation[:, :, i, i + 1] = 1.0

    verti_translation = torch.zeros((1, num_class, height, height), device=device, dtype=dtype)
    for j in range(height - 1):
        verti_translation[:, :, j, j + 1] = 1.0

    return hori_translation, verti_translation


def repeat_translation_matrices(hori_translation, verti_translation, batch_size):
    return hori_translation.repeat(batch_size, 1, 1, 1), verti_translation.repeat(batch_size, 1, 1, 1)


def report_pair(name, left, right):
    abs_diff = (left - right).abs()
    print(
        f"{name}: max_abs_diff={abs_diff.max().item():.8f}, "
        f"mean_abs_diff={abs_diff.mean().item():.8f}, "
        f"allclose={torch.allclose(left, right, atol=1e-6, rtol=1e-5)}"
    )


def _patched_shift_map(x, hori_translation, verti_translation, dy, dx):
    if hori_translation.dim() == 4:
        hori_translation = hori_translation.view(-1, hori_translation.shape[-2], hori_translation.shape[-1])
    if verti_translation.dim() == 4:
        verti_translation = verti_translation.view(-1, verti_translation.shape[-2], verti_translation.shape[-1])

    out = x

    if dy > 0:
        for _ in range(dy):
            out = torch.bmm(verti_translation.transpose(2, 1), out)
    elif dy < 0:
        for _ in range(-dy):
            out = torch.bmm(verti_translation, out)

    if dx > 0:
        for _ in range(dx):
            out = torch.bmm(out, hori_translation)
    elif dx < 0:
        for _ in range(-dx):
            out = torch.bmm(out, hori_translation.transpose(2, 1))

    return out


connect_loss_module.shift_map = _patched_shift_map


canonical_offsets = [
    (1, 1),
    (1, 0),
    (1, -1),
    (0, 1),
    (0, -1),
    (-1, 1),
    (-1, 0),
    (-1, -1),
]
canonical_names = [
    "down_right",
    "down",
    "down_left",
    "right",
    "left",
    "up_right",
    "up",
    "up_left",
]

kxk_offsets_conn3 = []
for dy in range(-1, 2):
    for dx in range(-1, 2):
        if dy == 0 and dx == 0:
            continue
        kxk_offsets_conn3.append((dy, dx))

canonical_to_kxk = torch.tensor(
    [kxk_offsets_conn3.index(offset) for offset in canonical_offsets],
    dtype=torch.long,
)
kxk_to_canonical = torch.argsort(canonical_to_kxk)

print("canonical order:")
for idx, (name, offset) in enumerate(zip(canonical_names, canonical_offsets)):
    print(f"  {idx}: {name:>10s} -> {offset}")

print("\nkxk conn_num=3 order:")
for idx, offset in enumerate(kxk_offsets_conn3):
    print(f"  {idx}: {offset}")

print("\ncanonical_to_kxk permutation:", canonical_to_kxk.tolist())
print("kxk_to_canonical permutation:", kxk_to_canonical.tolist())

canonical order:
  0: down_right -> (1, 1)
  1:       down -> (1, 0)
  2:  down_left -> (1, -1)
  3:      right -> (0, 1)
  4:       left -> (0, -1)
  5:   up_right -> (-1, 1)
  6:         up -> (-1, 0)
  7:    up_left -> (-1, -1)

kxk conn_num=3 order:
  0: (-1, -1)
  1: (-1, 0)
  2: (-1, 1)
  3: (0, -1)
  4: (0, 1)
  5: (1, -1)
  6: (1, 0)
  7: (1, 1)

canonical_to_kxk permutation: [7, 6, 5, 4, 3, 2, 1, 0]
kxk_to_canonical permutation: [7, 6, 5, 4, 3, 2, 1, 0]


In [33]:
# 3. 테스트 입력 생성 (재현 가능한 랜덤 텐서 + 이동 행렬)
B, C, H, W = 2, 1, 6, 7

small_batch = B
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hori_translation, verti_translation = build_translation_matrices(H, W, num_class=C, device=device)
hori_translation_batched, verti_translation_batched = repeat_translation_matrices(
    hori_translation,
    verti_translation,
    small_batch,
)

# Canonical semantic order matches Bilateral_voting.
affinity_canonical = torch.rand((B, C, 8, H, W), device=device, dtype=torch.float32)
affinity_kxk_raw = affinity_canonical.clone()
affinity_kxk_aligned = affinity_canonical.index_select(2, canonical_to_kxk.to(device))

print_tensor_info("affinity_canonical", affinity_canonical)
print_tensor_info("hori_translation", hori_translation_batched)
print_tensor_info("verti_translation", verti_translation_batched)
print("affinity_kxk_raw order equals canonical order:", torch.equal(affinity_kxk_raw, affinity_canonical))
print("affinity_kxk_aligned permutation:", canonical_to_kxk.tolist())

affinity_canonical: shape=(2, 1, 8, 6, 7), device=cuda:0, dtype=torch.float32, min=0.001029, max=0.999726, mean=0.497907
hori_translation: shape=(2, 1, 7, 7), device=cuda:0, dtype=torch.float32, min=0.000000, max=1.000000, mean=0.122449
verti_translation: shape=(2, 1, 6, 6), device=cuda:0, dtype=torch.float32, min=0.000000, max=1.000000, mean=0.138889
affinity_kxk_raw order equals canonical order: True
affinity_kxk_aligned permutation: [7, 6, 5, 4, 3, 2, 1, 0]


In [34]:
# 4. Bilateral_voting vs Bilateral_voting_kxk(conn_num=3) 직접 비교
with torch.no_grad():
    pred_bv, vote_bv = Bilateral_voting(affinity_canonical, hori_translation_batched, verti_translation_batched)
    pred_kxk_raw, vote_kxk_raw = Bilateral_voting_kxk(
        affinity_kxk_raw,
        hori_translation_batched,
        verti_translation_batched,
        conn_num=3,
    )

report_pair("pred_mask (raw)", pred_bv, pred_kxk_raw)
report_pair("vote_out (raw)", vote_bv, vote_kxk_raw)

channel_abs_diff = (vote_bv - vote_kxk_raw).abs().amax(dim=(0, 1, 3, 4))
print("per-channel max abs diff (raw order):")
for idx, value in enumerate(channel_abs_diff.tolist()):
    print(f"  channel {idx}: {value:.8f}")

pred_mask (raw): max_abs_diff=0.56026083, mean_abs_diff=0.18525384, allclose=False
vote_out (raw): max_abs_diff=0.98775280, mean_abs_diff=0.19333023, allclose=False
per-channel max abs diff (raw order):
  channel 0: 0.73751909
  channel 1: 0.94204724
  channel 2: 0.81057888
  channel 3: 0.85403401
  channel 4: 0.76016784
  channel 5: 0.98775280
  channel 6: 0.90389770
  channel 7: 0.88970810


In [27]:
# 5. 채널 정렬 적용 후 결과 재비교
with torch.no_grad():
    pred_kxk_aligned, vote_kxk_aligned = Bilateral_voting_kxk(
        affinity_kxk_aligned,
        hori_translation_batched,
        verti_translation_batched,
        conn_num=3,
    )

vote_kxk_aligned_to_canonical = vote_kxk_aligned.index_select(2, kxk_to_canonical.to(device))

report_pair("pred_mask (aligned)", pred_bv, pred_kxk_aligned)
report_pair("vote_out (aligned, reordered to canonical)", vote_bv, vote_kxk_aligned_to_canonical)

print("aligned vote_out matches canonical order:", torch.allclose(vote_bv, vote_kxk_aligned_to_canonical, atol=1e-6, rtol=1e-5))

pred_mask (aligned): max_abs_diff=0.00000000, mean_abs_diff=0.00000000, allclose=True
vote_out (aligned, reordered to canonical): max_abs_diff=0.00000000, mean_abs_diff=0.00000000, allclose=True
aligned vote_out matches canonical order: True


In [29]:
# 6. 다중 케이스 자동 검증 (shape/seed 반복)
# 7. 오차 통계 및 단정문(Assertion) 리포트
case_specs = [
    dict(case_id="case_01", seed=0, B=1, C=1, H=5, W=5),
    dict(case_id="case_02", seed=1, B=2, C=1, H=6, W=7),
    dict(case_id="case_03", seed=2, B=2, C=2, H=8, W=6),
    dict(case_id="case_04", seed=3, B=3, C=1, H=7, W=8),
]

results = []
raw_pass_count = 0
aligned_pass_count = 0

for spec in case_specs:
    torch.manual_seed(spec["seed"])
    np.random.seed(spec["seed"])
    random.seed(spec["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(spec["seed"])

    case_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    hori_t, verti_t = build_translation_matrices(spec["H"], spec["W"], num_class=spec["C"], device=case_device)
    hori_t_batched, verti_t_batched = repeat_translation_matrices(hori_t, verti_t, spec["B"])
    affinity = torch.rand((spec["B"], spec["C"], 8, spec["H"], spec["W"]), device=case_device, dtype=torch.float32)
    affinity_aligned = affinity.index_select(2, canonical_to_kxk.to(case_device))

    with torch.no_grad():
        pred_bv_case, vote_bv_case = Bilateral_voting(affinity, hori_t_batched, verti_t_batched)
        pred_kxk_raw_case, vote_kxk_raw_case = Bilateral_voting_kxk(affinity, hori_t_batched, verti_t_batched, conn_num=3)
        pred_kxk_aligned_case, vote_kxk_aligned_case = Bilateral_voting_kxk(
            affinity_aligned, hori_t_batched, verti_t_batched, conn_num=3
        )

    vote_kxk_aligned_case = vote_kxk_aligned_case.index_select(2, kxk_to_canonical.to(case_device))

    raw_pred_ok = torch.allclose(pred_bv_case, pred_kxk_raw_case, atol=1e-6, rtol=1e-5)
    raw_vote_ok = torch.allclose(vote_bv_case, vote_kxk_raw_case, atol=1e-6, rtol=1e-5)
    aligned_pred_ok = torch.allclose(pred_bv_case, pred_kxk_aligned_case, atol=1e-6, rtol=1e-5)
    aligned_vote_ok = torch.allclose(vote_bv_case, vote_kxk_aligned_case, atol=1e-6, rtol=1e-5)

    raw_pred_abs = (pred_bv_case - pred_kxk_raw_case).abs()
    raw_vote_abs = (vote_bv_case - vote_kxk_raw_case).abs()
    aligned_pred_abs = (pred_bv_case - pred_kxk_aligned_case).abs()
    aligned_vote_abs = (vote_bv_case - vote_kxk_aligned_case).abs()

    raw_pass_count += int(raw_pred_ok and raw_vote_ok)
    aligned_pass_count += int(aligned_pred_ok and aligned_vote_ok)

    results.append({
        "case_id": spec["case_id"],
        "seed": spec["seed"],
        "shape": f'({spec["B"]}, {spec["C"]}, 8, {spec["H"]}, {spec["W"]})',
        "raw_pred_max": raw_pred_abs.max().item(),
        "raw_pred_mean": raw_pred_abs.mean().item(),
        "raw_vote_max": raw_vote_abs.max().item(),
        "raw_vote_mean": raw_vote_abs.mean().item(),
        "aligned_pred_max": aligned_pred_abs.max().item(),
        "aligned_pred_mean": aligned_pred_abs.mean().item(),
        "aligned_vote_max": aligned_vote_abs.max().item(),
        "aligned_vote_mean": aligned_vote_abs.mean().item(),
        "raw_pred_ok": raw_pred_ok,
        "raw_vote_ok": raw_vote_ok,
        "aligned_pred_ok": aligned_pred_ok,
        "aligned_vote_ok": aligned_vote_ok,
    })

header = [
    "case_id",
    "seed",
    "shape",
    "raw_pred_max",
    "raw_vote_max",
    "aligned_pred_max",
    "aligned_vote_max",
    "raw_ok",
    "aligned_ok",
]
print(" | ".join(header))
print("-" * 120)
for row in results:
    raw_ok = row["raw_pred_ok"] and row["raw_vote_ok"]
    aligned_ok = row["aligned_pred_ok"] and row["aligned_vote_ok"]
    print(
        " | ".join(
            [
                str(row["case_id"]),
                str(row["seed"]),
                row["shape"],
                f"{row['raw_pred_max']:.8f}",
                f"{row['raw_vote_max']:.8f}",
                f"{row['aligned_pred_max']:.8f}",
                f"{row['aligned_vote_max']:.8f}",
                str(raw_ok),
                str(aligned_ok),
            ]
        )
    )

print(f"raw pass count: {raw_pass_count}/{len(results)}")
print(f"aligned pass count: {aligned_pass_count}/{len(results)}")

for row in results:
    if not (row["aligned_pred_ok"] and row["aligned_vote_ok"]):
        raise AssertionError(
            f"Aligned comparison failed for {row['case_id']}: "
            f"pred_max={row['aligned_pred_max']:.8f}, pred_mean={row['aligned_pred_mean']:.8f}, "
            f"vote_max={row['aligned_vote_max']:.8f}, vote_mean={row['aligned_vote_mean']:.8f}"
        )

assert aligned_pass_count == len(results), f"Expected all aligned cases to pass, got {aligned_pass_count}/{len(results)}"
print("All aligned cases match within atol=1e-6, rtol=1e-5.")

case_id | seed | shape | raw_pred_max | raw_vote_max | aligned_pred_max | aligned_vote_max | raw_ok | aligned_ok
------------------------------------------------------------------------------------------------------------------------
case_01 | 0 | (1, 1, 8, 5, 5) | 0.62519479 | 0.78228801 | 0.00000000 | 0.00000000 | False | True
case_02 | 1 | (2, 1, 8, 6, 7) | 0.72810173 | 0.92881197 | 0.00000000 | 0.00000000 | False | True
case_03 | 2 | (2, 2, 8, 8, 6) | 0.71981579 | 0.99045920 | 0.00000000 | 0.00000000 | False | True
case_04 | 3 | (3, 1, 8, 7, 8) | 0.63942462 | 0.90817130 | 0.00000000 | 0.00000000 | False | True
raw pass count: 0/4
aligned pass count: 4/4
All aligned cases match within atol=1e-6, rtol=1e-5.


# 8. 5x5 채널 순서 검증

`conn_num=5`에서 `Bilateral_voting_kxk` 채널 정책을 확인합니다.
- `0~7`: 8-connectivity (`Bilateral_voting`와 동일 의미)
- `8~23`: 나머지 16개 오프셋

In [35]:
# 8-1. conn_num=5 오프셋 순서 생성 및 검증
radius5 = 2

kxk_offsets_conn5 = [o for o in canonical_offsets if abs(o[0]) <= radius5 and abs(o[1]) <= radius5]
for dy in range(-radius5, radius5 + 1):
    for dx in range(-radius5, radius5 + 1):
        if dy == 0 and dx == 0:
            continue
        if (dy, dx) not in kxk_offsets_conn5:
            kxk_offsets_conn5.append((dy, dx))

print(f"conn_num=5 total channels: {len(kxk_offsets_conn5)}")
print("first 8 offsets (must match canonical 8-connectivity):")
for i in range(8):
    print(f"  {i}: {kxk_offsets_conn5[i]} == {canonical_offsets[i]}")

print("remaining offsets 8~23:")
for i, o in enumerate(kxk_offsets_conn5[8:], start=8):
    print(f"  {i}: {o}")

assert len(kxk_offsets_conn5) == 24, f"Expected 24 channels for conn_num=5, got {len(kxk_offsets_conn5)}"
assert kxk_offsets_conn5[:8] == canonical_offsets, "Channels 0~7 are not aligned with Bilateral_voting 8-connectivity order"
print("conn_num=5 ordering check passed: 0~7 aligned, 8~23 remaining offsets.")

conn_num=5 total channels: 24
first 8 offsets (must match canonical 8-connectivity):
  0: (1, 1) == (1, 1)
  1: (1, 0) == (1, 0)
  2: (1, -1) == (1, -1)
  3: (0, 1) == (0, 1)
  4: (0, -1) == (0, -1)
  5: (-1, 1) == (-1, 1)
  6: (-1, 0) == (-1, 0)
  7: (-1, -1) == (-1, -1)
remaining offsets 8~23:
  8: (-2, -2)
  9: (-2, -1)
  10: (-2, 0)
  11: (-2, 1)
  12: (-2, 2)
  13: (-1, -2)
  14: (-1, 2)
  15: (0, -2)
  16: (0, 2)
  17: (1, -2)
  18: (1, 2)
  19: (2, -2)
  20: (2, -1)
  21: (2, 0)
  22: (2, 1)
  23: (2, 2)
conn_num=5 ordering check passed: 0~7 aligned, 8~23 remaining offsets.


In [38]:
# 8-2. 5x5 수치 검증
# Case A: channels 8~23 = 0 이면, Bilateral_voting_kxk(conn_num=5)의 pred/vote(0~7)는 Bilateral_voting와 일치해야 함
B5, C5, H5, W5 = 2, 1, 8, 9
device5 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

h5, v5 = build_translation_matrices(H5, W5, num_class=C5, device=device5)
h5b, v5b = repeat_translation_matrices(h5, v5, B5)

affinity5 = torch.zeros((B5, C5, 24, H5, W5), device=device5, dtype=torch.float32)
affinity5[:, :, :8] = torch.rand((B5, C5, 8, H5, W5), device=device5)

with torch.no_grad():
    pred_bv5, vote_bv5 = Bilateral_voting(affinity5[:, :, :8], h5b, v5b)
    pred_kxk5, vote_kxk5 = Bilateral_voting_kxk(affinity5, h5b, v5b, conn_num=5)

report_pair("5x5 CaseA pred_mask", pred_bv5, pred_kxk5)
report_pair("5x5 CaseA vote_out first8", vote_bv5, vote_kxk5[:, :, :8])

assert torch.allclose(pred_bv5, pred_kxk5, atol=1e-6, rtol=1e-5), "5x5 CaseA pred mismatch"
assert torch.allclose(vote_bv5, vote_kxk5[:, :, :8], atol=1e-6, rtol=1e-5), "5x5 CaseA first8 vote mismatch"

# Case B: full 24 channels 랜덤 입력에서 first8 채널 의미가 보존되는지 확인
# (pred는 24채널 max를 사용하므로 Bilateral_voting pred와 직접 동일 비교 대상이 아님)
affinity5_full = torch.rand((B5, C5, 24, H5, W5), device=device5, dtype=torch.float32)
with torch.no_grad():
    _, vote_kxk5_full = Bilateral_voting_kxk(affinity5_full, h5b, v5b, conn_num=5)

print_tensor_info("5x5 full vote_out", vote_kxk5_full)
print("5x5 tests passed.")

5x5 CaseA pred_mask: max_abs_diff=0.00000000, mean_abs_diff=0.00000000, allclose=True
5x5 CaseA vote_out first8: max_abs_diff=0.00000000, mean_abs_diff=0.00000000, allclose=True
5x5 full vote_out: shape=(2, 1, 24, 8, 9), device=cuda:0, dtype=torch.float32, min=0.000000, max=0.980655, mean=0.174381
5x5 tests passed.
